# SDG mentions in the Daily Briefings

This notebook visualize the mentions of the 17SDGs in the UN daily briefings.
If an SDG is mentioned is determined by an LLM giving the definition of each 
SDG and saved to a database.

SDG definitions are taken from https://sdgs.un.org/goals

## Set up

Download data to use this notebook
```bash
# Download briefings
briefings update 2026-08-01 2026-08-07

# Initialize the database
briefings init-db

# Load the briefings to the database
briefings load 2026-08-01 2026-08-07
```

In [1]:
from datetime import datetime

from un_llm_analysis.database import (
    get_db_connection,
    get_sdgs_between_dates,
)
from un_llm_analysis.analysis.sdg import extract_sdg_mentions_from_sections

In [ ]:
# Parameters
start_date = "2026-08-01"
end_date = "2026-08-07"
model = "llama3.2"

# Database connection
conn = get_db_connection()

# Extract SDG from sections and insert them into the database
extract_sdg_mentions_from_sections(
    conn=conn, start_date=start_date, end_date=end_date, model=model, overwrite=False
)

# Visualisation

In [3]:
import plotly.express as px

conn = get_db_connection()
llm = "llama3.2"
df = get_sdgs_between_dates(conn=conn, start_date=start_date, end_date=end_date)

df["order"] = df.sdg.apply(lambda x: int(str(x).replace("SDG", "")))
df.sort_values(by="order", inplace=True)

# Prepare and sort data for plot
df_plot = (
    df[df["model"] == llm][["order", "sdg"]]
    .value_counts()
    .reset_index()
    .sort_values("order", ascending=False)
    .set_index("sdg")
    .drop(columns=["order"])
)

x = [x[0] for x in df_plot.values.tolist()]  # frequency of each SDG in the sections.
y = (
    df_plot.index.tolist()
)  # values from the "sdg" column with values: ["SDG1", "SDG2", ..., "SDG17"]

fig = px.bar(
    df,
    x=x,
    y=y,
    labels={
        "x": "# of Mentions in Sections",
        "y": "Sustainable Development Goal",
    },
    orientation="h",
    title=f"17SDGs in {len(df.briefing_id.unique())} Daily Briefings between {str(datetime.strptime(str(df.briefing_id.min()), "%Y%m%d"))[:10]} and {str(datetime.strptime(str(df.briefing_id.max()), "%Y%m%d"))[:10]}",
)
fig.show()